In [ ]:
#── Cell 1: Verify GPU is available ──────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f"✓ GPU ready: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ No GPU detected — go to Runtime → Change runtime type → T4 GPU")

✓ GPU ready: Tesla T4
  VRAM: 15.6 GB


In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q sentence-transformers tqdm numpy

In [ ]:
# ── Cell 4: Full precompute code ─────────────────────────────────────────────
# All logic from precompute.py — adapted for Colab paths and GPU batch size

import json
import gzip
import os
from datetime import date, datetime
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR      = Path('/content/data')
ARTIFACTS_DIR = Path('/content/artifacts')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Model config ─────────────────────────────────────────────────────────────
EMBEDDING_MODEL  = 'BAAI/bge-small-en-v1.5'
EMBED_BATCH_SIZE = 512   # GPU can handle larger batches than CPU (was 256)
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

# ── Constants (identical to precompute.py) ───────────────────────────────────
MAX_EXPERT_ZERO_DURATION_RATIO   = 0.7
MIN_EXPERT_SKILLS_FOR_CHECK      = 5
CAREER_DURATION_OVERAGE_FACTOR   = 1.4
CAREER_DURATION_TOLERANCE_MONTHS = 24

CONSULTING_INDUSTRIES = {
    'it services', 'consulting', 'professional services',
    'outsourcing', 'staffing', 'managed services', 'bpo',
    'information technology and services',
}
CONSULTING_FIRMS_FALLBACK = {
    'tcs', 'tata consultancy', 'infosys', 'wipro', 'accenture',
    'cognizant', 'capgemini', 'hcl', 'tech mahindra', 'mphasis',
    'hexaware', 'ltimindtree', 'mindtree', 'persistent', 'coforge',
    'kpit', 'niit', 'zensar', 'birlasoft', 'mastech',
}
RESEARCH_INDUSTRY_KEYWORDS = {
    'research', 'lab', 'labs', 'university', 'academia',
    'institute', 'academic', 'r&d',
}
PRODUCT_INDUSTRY_KEYWORDS = {
    'product', 'startup', 'saas', 'platform', 'app', 'software',
    'technology', 'internet', 'fintech', 'edtech', 'healthtech',
}
RELEVANT_TITLE_KEYWORDS = {
    'machine learning', 'ml engineer', 'ai engineer', 'nlp engineer',
    'data scientist', 'research engineer', 'applied scientist',
    'research scientist', 'computer vision',
    'software engineer', 'backend engineer', 'sde', 'swe',
    'data engineer', 'platform engineer', 'search engineer',
    'full stack', 'fullstack', 'analytics engineer',
    'data analyst', 'developer', 'programmer',
}
IRRELEVANT_TITLE_KEYWORDS = {
    'marketing', 'operations manager', 'accountant', 'accounting',
    'civil engineer', 'mechanical engineer', 'hr manager', 'human resources',
    'customer support', 'customer service', 'sales', 'brand',
    'content writer', 'seo', 'recruiter', 'talent acquisition',
    'supply chain', 'logistics', 'finance manager', 'legal',
    'graphic design', 'ui designer',
}
PROD_KEYWORDS_GENERAL = {
    'deployed', 'production', 'shipped', 'launched', 'scaled',
    'serving', 'pipeline', 'real-time', 'users', 'api',
    'microservice', 'kubernetes', 'docker', 'ci/cd', 'monitoring',
}
PROD_KEYWORDS_AI = {
    'embedding', 'embeddings', 'vector', 'retrieval', 'ranking',
    'model', 'inference', 'fine-tuned', 'fine-tuning', 'training',
    'rag', 'transformer', 'llm', 'nlp', 'recommendation',
    'similarity', 'index', 'search', 'rerank', 'faiss',
    'sentence-transformer', 'bert', 'gpt', 'neural',
}
PREFERRED_LOCATIONS = {
    'pune', 'noida', 'delhi', 'gurugram', 'gurgaon', 'ncr',
    'hyderabad', 'mumbai', 'bangalore', 'bengaluru',
}
CV_SPEECH_TITLES = ['computer vision','vision engineer','speech','asr',
                    'robotics','object detection']
CV_SPEECH_SKILLS_ADV = ['opencv','yolo','object detection','image classification',
                         'asr','tts','speech recognition','diffusion model']
FICTIONAL_COMPANIES = {'dunder mifflin', 'stark industries', 'wayne enterprises',
                       'acme corp', 'initech', 'hooli', 'globex inc', 'umbrella corp'}
TODAY = date.today()

print('Constants loaded.')

Using device: cuda
Constants loaded.


In [ ]:
# Fix — move candidates.jsonl into /content/data/ where the code expects it
import shutil, os
from pathlib import Path

Path('/content/data').mkdir(exist_ok=True)

if Path('/content/candidates.jsonl').exists():
    shutil.move('/content/candidates.jsonl', '/content/data/candidates.jsonl')
    print('✓ Moved to /content/data/candidates.jsonl')
else:
    # Check where it actually is
    import subprocess
    result = subprocess.run(['find', '/content', '-name', 'candidates*', '-type', 'f'], capture_output=True, text=True)
    print('Found at:', result.stdout)

Found at: /content/data/candidates.jsonl



In [ ]:
# ── Cell 5: Helper functions (identical logic to precompute.py) ───────────────

def load_candidates(data_dir: Path) -> list:
    data_dir = Path(data_dir)
    candidates_jsonl = data_dir/'candidates.jsonl'

    if candidates_jsonl.exists():
        print(f'Loading from {candidates_jsonl} ...')
        with open(candidates_jsonl, 'rt', encoding='utf-8') as f:
            candidates = [json.loads(line) for line in f if line.strip()]
    else:
        raise FileNotFoundError(f'No candidate file found in {data_dir}')

    print(f'Loaded {len(candidates):,} candidates')
    return candidates

def domain_fit_score(c: dict) -> float:
    career = c.get('career_history', [])
    skills = c.get('skills', [])
    titles = [r.get('title','').lower() for r in career]

    # If majority of career titles are CV/speech → hard penalty
    cv_title_hits = sum(1 for t in titles
                        if any(kw in t for kw in CV_SPEECH_TITLES))
    if career and cv_title_hits / len(career) >= 0.6:
        return 0.25  # kills rank-5 Sarvam CV Engineer, rank-2 Berlin CV Eng

    # If 2+ expert/advanced CV skills AND fewer IR skills → moderate penalty
    expert_skills = [s['name'].lower() for s in skills
                     if s.get('proficiency') in ('expert','advanced')]
    cv_hits = sum(1 for s in expert_skills
                  if any(kw in s for kw in CV_SPEECH_SKILLS_ADV))
    ir_hits = sum(1 for s in expert_skills
                  if any(kw in s for kw in
                         ['retrieval','search','ranking','embedding',
                          'vector','nlp','bert','transformer','information retrieval']))
    if cv_hits >= 2 and ir_hits < cv_hits:
        return 0.5

    return 1.0
def yoe_score(years: float) -> float:
    """Tent function: peaks at 5-9 years (JD sweet spot), penalizes over/under."""
    if years < 2:    return 0.10   # too junior
    if years < 4:    return 0.40   # junior
    if years < 5:    return 0.70   # slightly under
    if years <= 9:   return 1.00   # sweet spot per JD
    if years <= 12:  return 0.60   # slightly over
    return 0.25

def is_honeypot(c: dict) -> bool:
    skills = c.get('skills', [])
    yoe    = c.get('profile', {}).get('years_of_experience', 0)
    expert_skills = [s for s in skills if s.get('proficiency') in ('expert', 'advanced')]
    if len(expert_skills) >= MIN_EXPERT_SKILLS_FOR_CHECK:
        zero_dur = [s for s in expert_skills if s.get('duration_months', 0) == 0]
        if len(zero_dur) / len(expert_skills) > MAX_EXPERT_ZERO_DURATION_RATIO:
            return True
    career = c.get('career_history', [])
    total_career_months = sum(r.get('duration_months', 0) for r in career)
    max_possible_months = yoe * 12 * CAREER_DURATION_OVERAGE_FACTOR
    if yoe > 0 and total_career_months > max_possible_months + CAREER_DURATION_TOLERANCE_MONTHS:
        return True

    career = c.get('career_history', [])
    companies = {r.get('company','').lower() for r in career}
    if companies & FICTIONAL_COMPANIES:
        return True
    return False


def build_embed_text(c: dict) -> str:
    parts = []

    # Career descriptions — these capture plain language Tier 5 signals
    for role in c.get('career_history', []):
        title = role.get('title', '').strip()
        desc  = role.get('description', '').strip()
        if title: parts.append(title)
        if desc:  parts.append(desc)

    # Only include skills that have real duration backing
    # This stops "LlamaIndex expert, 91mo" from saving a CV engineer
    # while preserving "Weaviate expert, 88mo" for a real IR engineer
    strong_skills = [
        s['name'] for s in c.get('skills', [])
        if s.get('proficiency') in ('expert', 'advanced')
        and s.get('duration_months', 0) >= 12  # ← at least 1yr of real use
    ]
    weak_skills = [
        s['name'] for s in c.get('skills', [])
        if s.get('proficiency') in ('expert', 'advanced')
        and s.get('duration_months', 0) < 12
    ]

    if strong_skills:
        # Strong skills get prominent placement — repeated for embedding weight
        parts.append('Core expertise: ' + ', '.join(strong_skills))
    if weak_skills:
        # Weak skills mentioned once, at the end, lower influence
        parts.append('Also familiar with: ' + ', '.join(weak_skills))

    return ' '.join(parts)[:2000]


def career_title_score(career: list) -> float:
    if not career: return 0.3
    titles = [r.get('title', '').lower() for r in career]
    if all(any(kw in t for kw in IRRELEVANT_TITLE_KEYWORDS) for t in titles):
        return 0.1
    for i, title in enumerate(titles):
        if any(kw in title for kw in RELEVANT_TITLE_KEYWORDS):
            return max(1.0 - i * 0.2, 0.4)
    return 0.4


def is_consulting_role(role: dict) -> bool:
    industry = role.get('industry', '').lower()
    company  = role.get('company',  '').lower()
    if industry in CONSULTING_INDUSTRIES: return True
    if any(firm in company for firm in CONSULTING_FIRMS_FALLBACK): return True
    return False


def location_match_score(c: dict) -> float:
    location = c.get('profile', {}).get('location', '').lower()
    willing  = c.get('redrob_signals', {}).get('willing_to_relocate', False)
    if any(city in location for city in PREFERRED_LOCATIONS): return 1.0
    return 0.7 if willing else 0.3


def skill_depth_score(c: dict) -> float:
    skills    = c.get('skills', [])
    high_prof = [s for s in skills if s.get('proficiency') in ('expert', 'advanced')]
    if not high_prof: return 0.5
    with_dur = [s for s in high_prof if s.get('duration_months', 0) > 0]
    return len(with_dur) / len(high_prof)


def extract_career_features(c: dict) -> np.ndarray:
    profile  = c.get('profile', {})
    career   = c.get('career_history', [])
    raw_yoe = profile.get('years_of_experience', 0)
    yoe     = yoe_score(raw_yoe)
    industries         = [r.get('industry', '').lower() for r in career]
    role_is_consulting = [is_consulting_role(r) for r in career]
    role_is_research   = [any(kw in ind for kw in RESEARCH_INDUSTRY_KEYWORDS) for ind in industries]
    has_product = any(
        any(kw in ind for kw in PRODUCT_INDUSTRY_KEYWORDS) or (not ic and not ir)
        for ind, ic, ir in zip(industries, role_is_consulting, role_is_research)
    )
    consulting_penalty = 0.3 if (all(role_is_consulting) and career) else 1.0
    research_penalty   = 0.3 if (all(role_is_research)   and career) else 1.0
    tenures    = [r.get('duration_months', 0) for r in career]
    domain_score = domain_fit_score(c)
    title_sc   = career_title_score(career)
    all_descs  = ' '.join(r.get('description', '').lower() for r in career)
    prod_signal    = min(sum(1 for kw in PROD_KEYWORDS_GENERAL if kw in all_descs) / 5.0, 1.0)
    ai_prod_signal = min(sum(1 for kw in PROD_KEYWORDS_AI    if kw in all_descs) / 5.0, 1.0)
    return np.array([
        yoe, float(has_product), consulting_penalty, research_penalty,
        domain_score, title_sc, prod_signal, ai_prod_signal,
        skill_depth_score(c), location_match_score(c),
    ], dtype=np.float32)


def extract_behavioral_features(c: dict) -> np.ndarray:
    sig = c.get('redrob_signals', {})
    try:
        last_active     = datetime.strptime(sig.get('last_active_date', ''), '%Y-%m-%d').date()
        months_inactive = (TODAY - last_active).days / 30.0
        if months_inactive <= 6:   recency = 1.0
        elif months_inactive >= 18: recency = 0.0
        else: recency = 1.0 - (months_inactive - 6) / 12.0
    except (ValueError, TypeError):
        recency = 0.5
    notice_score = max(0.0, 1.0 - sig.get('notice_period_days', 90) / 120.0)
    return np.array([
        recency,
        float(sig.get('open_to_work_flag', False)),
        float(sig.get('recruiter_response_rate', 0.0)),
        float(sig.get('interview_completion_rate', 0.0)),
        sig.get('profile_completeness_score', 0) / 100.0,
        max(sig.get('github_activity_score', -1), 0) / 100.0,
        min(sig.get('saved_by_recruiters_30d', 0), 20) / 20.0,
        notice_score,
        min(sig.get('applications_submitted_30d', 0), 10) / 10.0,
        (float(sig.get('verified_email', False)) +
         float(sig.get('verified_phone', False)) +
         float(sig.get('linkedin_connected', False))) / 3.0,
    ], dtype=np.float32)


print('All helper functions defined.')

All helper functions defined.


In [ ]:
# Step 1: Load
print('[1/6] Loading candidates...')
candidates = load_candidates(DATA_DIR)
n = len(candidates)

# Step 4: Career features
print('\n[4/6] Extracting career features...')
career_features = np.stack([extract_career_features(c) for c in tqdm(candidates)]).astype(np.float32)
np.save(ARTIFACTS_DIR / 'career_features.npy', career_features)
print(f'  Shape: {career_features.shape}')



[1/6] Loading candidates...
Loading from /content/data/candidates.jsonl ...
Loaded 100,000 candidates

[4/6] Extracting career features...


100%|██████████| 100000/100000 [00:10<00:00, 9963.59it/s]


  Shape: (100000, 10)


In [ ]:
import json
import numpy as np
from pathlib import Path

A = Path('/content/artifacts')

cf    = np.load(A / 'career_features.npy')
print(f'career_features    : {cf.shape}  — min={cf.min():.3f} max={cf.max():.3f}')
assert cf.shape[1]  == 10, f'Expected 10 career dims, got {cf.shape[1]}'
assert cf.min()  >= 0.0 and cf.max()  <= 1.01, 'Career features out of range'

from google.colab import files

artifact_files = [
    'career_features.npy'
]

for fname in artifact_files:
    fpath = ARTIFACTS_DIR / fname
    if fpath.exists():
        print(f'Downloading {fname} ({fpath.stat().st_size / 1e6:.2f} MB)...')
        files.download(str(fpath))
    else:
        print(f'⚠ Missing: {fname} — did Cell 6 complete successfully?')

career_features    : (100000, 10)  — min=0.000 max=1.000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── Cell 6: Run precompute pipeline ──────────────────────────────────────────

# Step 1: Load
print('[1/6] Loading candidates...')
candidates = load_candidates(DATA_DIR)
n = len(candidates)

# Step 2: IDs
print('\n[2/6] Extracting candidate IDs...')
candidate_ids = [c['candidate_id'] for c in candidates]
with open(ARTIFACTS_DIR / 'candidate_ids.json', 'w') as f:
    json.dump(candidate_ids, f)
print(f'  Saved {n:,} IDs')

# Step 3: Honeypots
print('\n[3/6] Detecting honeypots...')
honeypot_flags = np.array([is_honeypot(c) for c in tqdm(candidates)], dtype=bool)
np.save(ARTIFACTS_DIR / 'honeypot_flags.npy', honeypot_flags)
n_hp = honeypot_flags.sum()
print(f'  Flagged {n_hp:,} honeypots ({n_hp/n*100:.2f}%)')

# Step 4: Career features
print('\n[4/6] Extracting career features...')
career_features = np.stack([extract_career_features(c) for c in tqdm(candidates)]).astype(np.float32)
np.save(ARTIFACTS_DIR / 'career_features.npy', career_features)
print(f'  Shape: {career_features.shape}')

# Step 5: Behavioral features
print('\n[5/6] Extracting behavioral features...')
behavioral_features = np.stack([extract_behavioral_features(c) for c in tqdm(candidates)]).astype(np.float32)
np.save(ARTIFACTS_DIR / 'behavioral_features.npy', behavioral_features)
print(f'  Shape: {behavioral_features.shape}')

# Step 6: Embeddings on GPU
print(f'\n[6/6] Building embeddings on {DEVICE.upper()}...')
print('  Loading model...')
model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

print('  Building embed texts...')
embed_texts = [build_embed_text(c) for c in tqdm(candidates)]

print(f'  Encoding {n:,} candidates (batch={EMBED_BATCH_SIZE})...')
embeddings = model.encode(
    embed_texts,
    batch_size=EMBED_BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
np.save(ARTIFACTS_DIR / 'embeddings.npy', embeddings)
print(f'  Shape: {embeddings.shape}')

# Summary
print('\n✓ Precompute complete!')
print(f'  Candidates  : {n:,}')
print(f'  Honeypots   : {n_hp:,} ({n_hp/n*100:.2f}%)')
print(f'  Career dims : {career_features.shape[1]}')
print(f'  Behav dims  : {behavioral_features.shape[1]}')
print(f'  Embed dims  : {embeddings.shape[1]}')
print('\nArtifact sizes:')
for f in sorted(ARTIFACTS_DIR.iterdir()):
    print(f'  {f.name:<35} {f.stat().st_size / 1e6:.2f} MB')

[1/6] Loading candidates...
Loading from /content/data/candidates.jsonl ...
Loaded 100,000 candidates

[2/6] Extracting candidate IDs...
  Saved 100,000 IDs

[3/6] Detecting honeypots...


100%|██████████| 100000/100000 [00:00<00:00, 230447.78it/s]


  Flagged 77,447 honeypots (77.45%)

[4/6] Extracting career features...


 89%|████████▉ | 88839/100000 [00:09<00:01, 9505.82it/s] 


KeyboardInterrupt: 

In [ ]:
# ── Cell 8: Sanity check (run after Cell 6, before downloading) ───────────────
# Verifies shapes and spot-checks a few values

import json
import numpy as np
from pathlib import Path

A = Path('/content/artifacts')

ids   = json.load(open(A / 'candidate_ids.json'))
hp    = np.load(A / 'honeypot_flags.npy')
cf    = np.load(A / 'career_features.npy')
bf    = np.load(A / 'behavioral_features.npy')
emb   = np.load(A / 'embeddings.npy')

n = len(ids)
print(f'candidate_ids      : {n:,} entries')
print(f'honeypot_flags     : {hp.shape}  — {hp.sum()} flagged ({hp.mean()*100:.2f}%)')
print(f'career_features    : {cf.shape}  — min={cf.min():.3f} max={cf.max():.3f}')
print(f'behavioral_features: {bf.shape}  — min={bf.min():.3f} max={bf.max():.3f}')
print(f'embeddings         : {emb.shape} — L2 norms (first 5): {np.linalg.norm(emb[:5], axis=1).round(4)}')

# All checks
assert len(ids) == hp.shape[0] == cf.shape[0] == bf.shape[0] == emb.shape[0], 'Row count mismatch!'
assert cf.shape[1]  == 10, f'Expected 10 career dims, got {cf.shape[1]}'
assert bf.shape[1]  == 10, f'Expected 10 behavioral dims, got {bf.shape[1]}'
assert emb.shape[1] == 384, f'Expected 384 embedding dims, got {emb.shape[1]}'
assert cf.min()  >= 0.0 and cf.max()  <= 1.01, 'Career features out of range'
assert bf.min()  >= 0.0 and bf.max()  <= 1.01, 'Behavioral features out of range'
# Embeddings should be L2-normalized — norms should be ~1.0
norms = np.linalg.norm(emb, axis=1)
assert norms.min() > 0.98 and norms.max() < 1.02, f'Embeddings not normalized! norms: {norms.min():.4f}–{norms.max():.4f}'

print('\n✓ All sanity checks passed — safe to download.')

candidate_ids      : 100,000 entries
honeypot_flags     : (100000,)  — 30 flagged (0.03%)
career_features    : (100000, 10)  — min=0.000 max=1.000
behavioral_features: (100000, 10)  — min=0.000 max=1.000
embeddings         : (100000, 384) — L2 norms (first 5): [1. 1. 1. 1. 1.]

✓ All sanity checks passed — safe to download.


In [ ]:
# ── Cell 7: Download all artifacts ───────────────────────────────────────────
# Run this cell to download all 5 artifacts to your local machine.
# Save them into: D:\India.run\resume_ranker\artifacts\

from google.colab import files

artifact_files = [
    'candidate_ids.json',
    'honeypot_flags.npy',
    'career_features.npy',
    'behavioral_features.npy',
    'embeddings.npy',
]

for fname in artifact_files:
    fpath = ARTIFACTS_DIR / fname
    if fpath.exists():
        print(f'Downloading {fname} ({fpath.stat().st_size / 1e6:.2f} MB)...')
        files.download(str(fpath))
    else:
        print(f'⚠ Missing: {fname} — did Cell 6 complete successfully?')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>